## 1. Installation et imports

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy

In [ ]:
import os, re, json, math, time, gc, random, hashlib, unicodedata, textwrap, platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("python", platform.python_version())

## 2. Configuration

In [ ]:
STEP2_DIR = Path("outputs/step2_index")
STEP3_DIR = Path("outputs/step3_dataset")
STEP4_DIR = Path("outputs/step4_qlora")
ADAPTER_DIR = STEP4_DIR / "adapter"
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step5_pipeline"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 300        # identique à l'étape 1 (baseline A)
BATCH_SIZE = 4              # identique à l'étape 1 ; le garder pour B et C
ENFORCE_CITATION = True     # une réponse sans citation valide est rejetée (traitée comme abstention) ; la sortie brute du modèle est conservée

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

## 3. Chargement des artefacts des étapes précédentes et contrôles de cohérence

In [ ]:
index_cfg = json.load(open(STEP2_DIR / "index_config.json", encoding="utf-8"))
rag_prompt = json.load(open(STEP3_DIR / "rag_prompt.json", encoding="utf-8"))
dataset_cfg = json.load(open(STEP3_DIR / "dataset_config.json", encoding="utf-8"))
chunks = load_jsonl(STEP2_DIR / "chunks.jsonl")
chunk_by_id = {c["chunk_id"]: c for c in chunks}
K = int(index_cfg["k_final"])
ABSTAIN_MARKER, ABSTAIN_ANSWER = rag_prompt["abstain_marker"], rag_prompt["abstain_answer"]

assert rag_prompt["k_context"] == K, f"k de l'étape 3 ({rag_prompt['k_context']}) ≠ k_final de l'étape 2 ({K}) : refaire l'étape 3 puis l'étape 4"
assert dataset_cfg["step2_config"]["embedder"] == index_cfg["embedder"] and dataset_cfg["step2_config"]["chunking"] == index_cfg["chunking"], \
    "le dataset de l'étape 3 a été construit avec un autre index que celui chargé ici"
assert len(chunks) == index_cfg["n_chunks"], "chunks.jsonl ne correspond pas à index_config.json"
print(f"{len(chunks)} chunks | embedder={index_cfg['embedder']} méthode={index_cfg['method']} k={K} | abstention : « {ABSTAIN_MARKER} »")